<a href="https://colab.research.google.com/github/kyungeunvoyage/MotionDualAxisVib/blob/main/Exploratory_study_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⬇️ [선택] Google Drive 마운트 (CSV가 구글드라이브에 있을 때만)
USE_DRIVE = False  # ← CSV가 Drive에 있으면 True 로 바꾸세요
DRIVE_CSV_PATH = "/content/drive/MyDrive/your_folder/your_raw.csv"  # ← 본인 경로로 수정

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# @title 📦 환경/임포트 (Colab 표준만 사용: numpy/pandas/matplotlib)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
from statistics import NormalDist

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 12


In [ ]:
# @title 🧾 데이터 불러오기 (CSV 경로 지정) + 컬럼 확인
# === CSV 경로 지정 ===
CSV_PATH = "/content/your_raw.csv"  # ← 로컬 업로드 시 /content 아래에 올려두고 수정
if 'USE_DRIVE' in globals() and USE_DRIVE:
    CSV_PATH = DRIVE_CSV_PATH

# === 필수 컬럼 ===
REQUIRED_COLS = ["participant_id", "side", "axis", "waveform", "correct"]

# === 읽기 ===
df = pd.read_csv(CSV_PATH)

# === 컬럼명 스펙 맞추기(대/소문자 섞여도 수습) ===
colmap = {c.lower(): c for c in df.columns}
# 최소한 필요한 컬럼이 있는지 체크 후 스펙으로 리네임
def pick(name):
    # 여러 변형 대응: participant / participant_id / pid
    alts = {
        "participant_id": ["participant_id", "participant", "pid"],
        "side": ["side", "stim_side"],
        "axis": ["axis"],
        "waveform": ["waveform", "wf", "waveform_id"],
        "correct": ["correct"]
    }[name]
    for a in alts:
        if a in colmap:
            return colmap[a]
    raise KeyError(f"'{name}'(이)가 CSV에 없습니다. 후보: {alts}")

rename_map = {
    pick("participant_id"): "participant_id",
    pick("side"): "side",
    pick("axis"): "axis",
    pick("waveform"): "waveform",
    pick("correct"): "correct",
}
df = df.rename(columns=rename_map)

# === 타입 정리 ===
df["waveform"] = pd.to_numeric(df["waveform"], errors="coerce").astype("Int64")
df["correct"] = pd.to_numeric(df["correct"], errors="coerce").astype(int)
df["side"] = df["side"].astype(str)
df["axis"] = df["axis"].astype(str)

# === 간단 sanity check ===
print(df.head())
print("\n조건 조합 개수:", df.groupby(["side","axis","waveform"]).size().shape[0])
print("참가자 수:", df["participant_id"].nunique())


In [ ]:
# @title 🧮 보조함수: Wilson 95% CI / d′ 변환(2AFC)
def wilson_ci(successes, n, alpha=0.05):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (np.nan, np.nan)
    z = NormalDist().inv_cdf(1 - alpha/2)
    phat = successes / n
    denom = 1 + z**2/n
    center = (phat + z**2/(2*n)) / denom
    half_width = (z * sqrt((phat*(1-phat)/n) + (z**2/(4*n**2)))) / denom
    return (center - half_width, center + half_width)

def dprime_from_pc(pc):
    """2AFC 민감도 변환: d' = sqrt(2) * z(Pc)  (극단값 클리핑)"""
    pc = float(min(1 - 1e-6, max(1e-6, pc)))
    return np.sqrt(2) * NormalDist().inv_cdf(pc)


In [ ]:
# @title 📊 집계: side × axis × waveform별 정확도, Wilson CI, d′
agg = (df.groupby(["side","axis","waveform"])
         .agg(n=("correct","size"), hits=("correct","sum"))
         .reset_index())
agg["acc"] = agg["hits"] / agg["n"]
agg[["ci_low","ci_high"]] = agg.apply(
    lambda r: pd.Series(wilson_ci(r["hits"], r["n"])), axis=1
)
agg["dprime"] = agg["acc"].apply(dprime_from_pc)

# 보고용 정렬
side_order = sorted(agg["side"].unique())
axis_order = ["Y","Z"] if set(agg["axis"].unique())==set(["Y","Z"]) else sorted(agg["axis"].unique())
agg = agg.sort_values(["side","axis","waveform"]).reset_index(drop=True)

agg.head()


In [ ]:
# @title 📈 Figure 1. Accuracy by waveform (side × axis) with 95% CI
fig, axs = plt.subplots(1, len(side_order), figsize=(6*len(side_order), 4), sharey=True)

if len(side_order) == 1:
    axs = [axs]

for i, side in enumerate(side_order):
    sub = agg[agg["side"]==side]
    for ax_str in axis_order:
        part = sub[sub["axis"]==ax_str]
        x = part["waveform"].astype(int).values
        y = part["acc"].values
        yerr = np.vstack([y - part["ci_low"].values, part["ci_high"].values - y])
        axs[i].errorbar(x, y, yerr=yerr, fmt="o-", capsize=4, label=f"{ax_str}-axis")
    axs[i].set_title(f"{side} side")
    axs[i].set_xlabel("Waveform")
    axs[i].set_ylim(0.4, 1.0)
    axs[i].grid(alpha=.3)
    axs[i].legend()
axs[0].set_ylabel("Accuracy (proportion correct)")
plt.suptitle("Accuracy by waveform (side × axis) with 95% Wilson CI", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# @title 🔥 Figure 2. d′ heatmap (waveform × condition)
# 피벗: rows=waveform, cols=side-axis
agg["cond"] = agg["side"] + "-" + agg["axis"]
pivot = agg.pivot_table(index="waveform", columns="cond", values="dprime")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
im = ax.imshow(pivot.values, aspect="auto", origin="lower")
cbar = plt.colorbar(im)
cbar.set_label("d′ (sensitivity)")

ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index.astype(int).tolist())
ax.set_xlabel("Condition (side-axis)")
ax.set_ylabel("Waveform")
ax.set_title("Sensitivity (d′) by waveform and condition")

# 격자선(보조)
ax.set_xticks(np.arange(-.5, pivot.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, pivot.shape[0], 1), minor=True)
ax.grid(which="minor", color="w", linestyle="--", linewidth=0.3, alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# @title 💾 결과 저장: 요약 CSV & 그림 파일
OUT_SUMMARY_CSV = "/content/summary_side-axis-waveform.csv"
FIG1_PATH = "/content/fig_accuracy_by_waveform.png"
FIG2_PATH = "/content/fig_dprime_heatmap.png"

agg.to_csv(OUT_SUMMARY_CSV, index=False)

# Figure 저장을 위해 한 번 더 그리기(같은 코드 재사용)
# Fig1
fig, axs = plt.subplots(1, len(side_order), figsize=(6*len(side_order), 4), sharey=True)
if len(side_order) == 1:
    axs = [axs]
for i, side in enumerate(side_order):
    sub = agg[agg["side"]==side]
    for ax_str in axis_order:
        part = sub[sub["axis"]==ax_str]
        x = part["waveform"].astype(int).values
        y = part["acc"].values
        yerr = np.vstack([y - part["ci_low"].values, part["ci_high"].values - y])
        axs[i].errorbar(x, y, yerr=yerr, fmt="o-", capsize=4, label=f"{ax_str}-axis")
    axs[i].set_title(f"{side} side")
    axs[i].set_xlabel("Waveform")
    axs[i].set_ylim(0.4, 1.0)
    axs[i].grid(alpha=.3)
    axs[i].legend()
axs[0].set_ylabel("Accuracy (proportion correct)")
plt.suptitle("Accuracy by waveform (side × axis) with 95% Wilson CI", y=1.03)
plt.tight_layout()
plt.savefig(FIG1_PATH, dpi=300)
plt.close()

# Fig2
pivot = agg.pivot_table(index="waveform", columns="cond", values="dprime")
fig, ax = plt.subplots(figsize=(7.5, 4.8))
im = ax.imshow(pivot.values, aspect="auto", origin="lower")
cbar = plt.colorbar(im)
cbar.set_label("d′ (sensitivity)")
ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index.astype(int).tolist())
ax.set_xlabel("Condition (side-axis)")
ax.set_ylabel("Waveform")
ax.set_title("Sensitivity (d′) by waveform and condition")
ax.set_xticks(np.arange(-.5, pivot.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, pivot.shape[0], 1), minor=True)
ax.grid(which="minor", color="w", linestyle="--", linewidth=0.3, alpha=0.6)
plt.tight_layout()
plt.savefig(FIG2_PATH, dpi=300)
plt.close()

print("Saved:")
print(" -", OUT_SUMMARY_CSV)
print(" -", FIG1_PATH)
print(" -", FIG2_PATH)


In [ ]:
# @title 🧪 [옵션] CSV가 아직 없을 때: 가상 데이터 생성기 (스킵 가능)
# 실데이터 없을 때 테스트용. 실행하면 /content/sim_raw.csv 를 생성.
GENERATE_SIM = False  # ← 필요 시 True
if GENERATE_SIM:
    rng = np.random.default_rng(0)
    parts = [f"P{idx:02d}" for idx in range(1, 25)]
    sides = ["ulnar","radial"]
    axes = ["Y","Z"]
    waveforms = [1,2,3,4,5,6]
    rows = []
    for pid in parts:
        for side in sides:
            for axis in axes:
                base = 0.55 + (0.05 if side=="ulnar" else 0.0) + (0.08 if axis=="Z" else 0.0)
                for wf in waveforms:
                    wf_bonus = (wf-3)*0.02
                    p = np.clip(base + wf_bonus + rng.normal(0,0.02), 0.4, 0.95)
                    for _ in range(40):
                        rows.append({
                            "participant_id": pid,
                            "side": side,
                            "axis": axis,
                            "waveform": wf,
                            "correct": int(rng.random() < p)
                        })
    sim = pd.DataFrame(rows)
    sim.to_csv("/content/sim_raw.csv", index=False)
    print("생성 완료: /content/sim_raw.csv")
